# Brain System - End-to-End Test

Complete flow: School setup -> Users -> Content -> Workflow -> Assignment -> Student Activity

### Services
| Service | Port |
|---------|------|
| role-permission-service | 8080 |
| auth-service | 8081 |
| tenant-service | 8082 |
| user-profile-service | 8083 |
| content-service | 8085 |
| content-workflow-service | 8086 |
| mindmap-service | 8087 |
| notes-service | 8088 |
| recall-service | 8091 |

## Phase 0: Setup & Helpers

In [34]:
import requests
import json
import uuid
from datetime import datetime, timedelta

# --- Service URLs ---
HOST = "http://localhost"
ROLE_PERM_URL  = f"{HOST}:8080"
AUTH_URL        = f"{HOST}:8081"
TENANT_URL      = f"{HOST}:8082"
PROFILE_URL     = f"{HOST}:8083"
WORKFLOW_URL    = f"{HOST}:8086"
NOTES_URL       = f"{HOST}:8088"
RECALL_URL      = f"{HOST}:8091"

# --- Global state ---
data = {
    "tenant_id": None,
    "roles": {},       # role_name -> role_id
    "users": {},       # key -> {userId, accessToken, email}
    "notes": {},       # key -> {id, versionId}
    "classes": {},     # key -> class_id
    "workflows": {},   # key -> {workflowId, reviewTaskId}
    "assignments": {}, # key -> assignment_id
    "results": [],     # test result log
}

# --- Helpers ---
def h(token=None, tenant_id=None, user_id=None):
    """Build request headers."""
    headers = {"Content-Type": "application/json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    if tenant_id:
        headers["X-Tenant-Id"] = str(tenant_id)
    if user_id:
        headers["X-User-Id"] = str(user_id)
    return headers

def ok(resp, label, expected=None):
    """Print response and track pass/fail."""
    expected = expected or [200, 201]
    status = resp.status_code
    passed = status in expected
    icon = "PASS" if passed else "FAIL"
    data["results"].append((label, icon, status))
    try:
        body = resp.json()
    except Exception:
        body = resp.text[:200]
    print(f"[{icon}] {label} -> {status}")
    if not passed or True:  # always show response
        print(json.dumps(body, indent=2) if isinstance(body, (dict, list)) else body)
    print()
    return body if passed else None

SUFFIX = uuid.uuid4().hex[:6]
print(f"Test run suffix: {SUFFIX}")
print("Helpers loaded.")

Test run suffix: 837cda
Helpers loaded.


## Phase 1: Create Tenant (School)

In [35]:
# Check if tenant already exists
resp = requests.get(f"{TENANT_URL}/v1/resolve", params={"tenantKey": f"testschool-{SUFFIX}"})
if resp.status_code == 200:
    data["tenant_id"] = resp.json()["id"]
    print(f"Tenant already exists: {data['tenant_id']}")
else:
    resp = requests.post(f"{TENANT_URL}/v1/tenants", json={
        "tenantKey": f"testschool-{SUFFIX}",
        "name": f"Test School {SUFFIX}"
    })
    body = ok(resp, "Create Tenant")
    if body:
        data["tenant_id"] = body["id"]

TID = data["tenant_id"]
print(f"\nTenant ID: {TID}")

[PASS] Create Tenant -> 200
{
  "id": "1e20511f-422b-40fc-b443-ebbf90762edf",
  "tenantKey": "testschool-837cda",
  "name": "Test School 837cda",
  "status": "ACTIVE",
  "createdAt": "2026-02-08T12:23:27.427480383Z",
  "updatedAt": "2026-02-08T12:23:27.427480383Z"
}


Tenant ID: 1e20511f-422b-40fc-b443-ebbf90762edf


## Phase 2: Setup Permissions & Roles

In [36]:
# 2a. Create permissions (check first, create only if missing)
permissions = [
    {"code": "NOTE:CREATE", "description": "Create notes", "resource": "NOTE", "action": "CREATE", "active": True},
    {"code": "NOTE:READ",   "description": "Read notes",   "resource": "NOTE", "action": "READ",   "active": True},
    {"code": "NOTE:UPDATE", "description": "Update notes", "resource": "NOTE", "action": "UPDATE", "active": True},
    {"code": "NOTE:DELETE", "description": "Delete notes", "resource": "NOTE", "action": "DELETE", "active": True},
]

for p in permissions:
    check = requests.get(f"{ROLE_PERM_URL}/permissions/{p['code']}")
    if check.status_code == 200:
        print(f"  Permission {p['code']}: already exists")
        continue
    resp = requests.post(f"{ROLE_PERM_URL}/permissions", json=p)
    status = "created" if resp.status_code in [200, 201] else f"error:{resp.status_code} - {resp.text[:80]}"
    print(f"  Permission {p['code']}: {status}")

print("Permissions setup done.")

  Permission NOTE:CREATE: already exists
  Permission NOTE:READ: already exists
  Permission NOTE:UPDATE: already exists
  Permission NOTE:DELETE: already exists
Permissions setup done.


In [37]:
# 2b. Create roles for this tenant (check first, create only if missing)
role_defs = [
    {"name": "TENANT_ADMIN",     "description": "School admin",     "active": True},
    {"name": "SUBJECT_TEACHER",  "description": "Subject teacher",  "active": True},
    {"name": "CONTENT_CREATOR",  "description": "Content creator",  "active": True},
    {"name": "STUDENT",          "description": "Student",          "active": True},
]

# Fetch existing roles first
resp = requests.get(f"{ROLE_PERM_URL}/tenants/{TID}/roles")
existing_roles = {}
if resp.status_code == 200:
    for role in resp.json():
        existing_roles[role["name"]] = role["id"]

for r in role_defs:
    if r["name"] in existing_roles:
        data["roles"][r["name"]] = existing_roles[r["name"]]
        print(f"  Role {r['name']}: already exists -> {existing_roles[r['name']]}")
        continue
    resp = requests.post(f"{ROLE_PERM_URL}/tenants/{TID}/roles", json=r)
    if resp.status_code in [200, 201]:
        data["roles"][r["name"]] = resp.json()["id"]
        print(f"  Role {r['name']}: created -> {resp.json()['id']}")
    else:
        print(f"  Role {r['name']}: error {resp.status_code} - {resp.text[:100]}")

print(f"\nRoles: {json.dumps(data['roles'], indent=2)}")

  Role TENANT_ADMIN: created -> 5
  Role SUBJECT_TEACHER: created -> 6
  Role CONTENT_CREATOR: created -> 7
  Role STUDENT: created -> 8

Roles: {
  "TENANT_ADMIN": 5,
  "SUBJECT_TEACHER": 6,
  "CONTENT_CREATOR": 7,
  "STUDENT": 8
}


In [38]:
# 2c. Grant permissions to roles (check first, grant only if missing)
grants = [
    ("TENANT_ADMIN", "NOTE:CREATE", "TENANT"),
    ("TENANT_ADMIN", "NOTE:READ",   "TENANT"),
    ("TENANT_ADMIN", "NOTE:UPDATE", "TENANT"),
    ("TENANT_ADMIN", "NOTE:DELETE", "TENANT"),
    ("SUBJECT_TEACHER", "NOTE:CREATE", "CLASS"),
    ("SUBJECT_TEACHER", "NOTE:READ",   "CLASS"),
    ("SUBJECT_TEACHER", "NOTE:UPDATE", "CLASS"),
    ("SUBJECT_TEACHER", "NOTE:DELETE", "CLASS"),
    ("CONTENT_CREATOR", "NOTE:CREATE", "TENANT"),
    ("CONTENT_CREATOR", "NOTE:READ",   "TENANT"),
    ("CONTENT_CREATOR", "NOTE:UPDATE", "TENANT"),
    ("STUDENT", "NOTE:READ", "OWN"),
]

# Pre-fetch existing grants per role to avoid duplicates
existing_grants = {}  # role_id -> set of (permissionCode, scopeCode)
for role_name, role_id in data["roles"].items():
    resp = requests.get(f"{ROLE_PERM_URL}/tenants/{TID}/roles/{role_id}/grants")
    if resp.status_code == 200:
        existing_grants[role_id] = {(g["permissionCode"], g["scopeCode"]) for g in resp.json()}
    else:
        existing_grants[role_id] = set()

for role_name, perm_code, scope in grants:
    role_id = data["roles"].get(role_name)
    if not role_id:
        print(f"  SKIP: Role {role_name} not found")
        continue
    if (perm_code, scope) in existing_grants.get(role_id, set()):
        print(f"  {role_name} <- {perm_code} ({scope}): already exists")
        continue
    resp = requests.post(
        f"{ROLE_PERM_URL}/tenants/{TID}/roles/{role_id}/grants",
        json={"permissionCode": perm_code, "scopeCode": scope}
    )
    status = "granted" if resp.status_code in [200, 201] else f"error:{resp.status_code}"
    print(f"  {role_name} <- {perm_code} ({scope}): {status}")

print("\nPermission grants done.")

  TENANT_ADMIN <- NOTE:CREATE (TENANT): granted
  TENANT_ADMIN <- NOTE:READ (TENANT): granted
  TENANT_ADMIN <- NOTE:UPDATE (TENANT): granted
  TENANT_ADMIN <- NOTE:DELETE (TENANT): granted
  SUBJECT_TEACHER <- NOTE:CREATE (CLASS): granted
  SUBJECT_TEACHER <- NOTE:READ (CLASS): granted
  SUBJECT_TEACHER <- NOTE:UPDATE (CLASS): granted
  SUBJECT_TEACHER <- NOTE:DELETE (CLASS): granted
  CONTENT_CREATOR <- NOTE:CREATE (TENANT): granted
  CONTENT_CREATOR <- NOTE:READ (TENANT): granted
  CONTENT_CREATOR <- NOTE:UPDATE (TENANT): granted
  STUDENT <- NOTE:READ (OWN): granted

Permission grants done.


## Phase 3: Create Users & Assign Roles

In [39]:
# 3a. Signup users (try login first to check if already exists)
user_defs = [
    ("admin",    f"admin_{SUFFIX}@test.com",    "Admin@123",   "Admin User",     "TENANT_ADMIN"),
    ("teacher",  f"teacher_{SUFFIX}@test.com",  "Teacher@123", "Mrs. Sharma",    "SUBJECT_TEACHER"),
    ("creator",  f"creator_{SUFFIX}@test.com",  "Creator@123", "Content Writer", "CONTENT_CREATOR"),
    ("student1", f"student1_{SUFFIX}@test.com", "Student@123", "Aarav Patel",    "STUDENT"),
    ("student2", f"student2_{SUFFIX}@test.com", "Student@123", "Priya Singh",    "STUDENT"),
]

for key, email, password, name, role_name in user_defs:
    # Check if user already exists by trying login
    login_resp = requests.post(f"{AUTH_URL}/auth/login", json={
        "tenantId": str(TID), "identifier": email, "password": password
    })
    if login_resp.status_code == 200:
        body = login_resp.json()
        data["users"][key] = {
            "userId": body["userId"],
            "accessToken": body.get("accessToken"),
            "email": email, "password": password, "name": name, "role": role_name,
        }
        print(f"  {key}: already exists -> {body['userId']}")
        continue

    resp = requests.post(f"{AUTH_URL}/auth/signup", json={
        "tenantId": str(TID), "email": email, "password": password,
        "name": name, "joinMethod": "SELF_SIGNUP"
    })
    if resp.status_code in [200, 201]:
        body = resp.json()
        data["users"][key] = {
            "userId": body["userId"],
            "accessToken": body.get("accessToken"),
            "email": email, "password": password, "name": name, "role": role_name,
        }
        print(f"  {key}: signed up -> {body['userId']}")
    else:
        print(f"  {key}: error {resp.status_code} - {resp.text[:100]}")

print(f"\nUsers created: {list(data['users'].keys())}")

  admin: already exists -> 05ecdf57-f283-45fc-b5d2-caedfaba135d
  teacher: already exists -> 61258019-de92-4578-a619-c2500355b642
  creator: already exists -> b78898d1-6bee-4f85-9b0f-940fda8b17d7
  student1: already exists -> f63898d8-0eb1-4e42-bd3a-5f2e86cf3676
  student2: already exists -> 2321b3e4-dc79-4366-8806-30c4de1e687a

Users created: ['admin', 'teacher', 'creator', 'student1', 'student2']


In [40]:
# 3b. Assign roles to users (check first, assign only if missing)
for key, info in data["users"].items():
    role_id = data["roles"].get(info["role"])
    if not role_id:
        print(f"  SKIP {key}: role {info['role']} not found")
        continue
    # Check if role already assigned
    check = requests.get(f"{ROLE_PERM_URL}/tenants/{TID}/users/{info['userId']}/roles")
    if check.status_code == 200:
        assigned_role_ids = {r["roleId"] for r in check.json()}
        if role_id in assigned_role_ids:
            print(f"  {key} <- {info['role']}: already assigned")
            continue
    resp = requests.post(
        f"{ROLE_PERM_URL}/tenants/{TID}/users/{info['userId']}/roles",
        json={"roleId": role_id, "scopeType": "TENANT", "scopeId": None, "status": "ACTIVE"},
        headers={"Content-Type": "application/json", "X-User-Id": "admin"}
    )
    status = "assigned" if resp.status_code in [200, 201] else f"error:{resp.status_code}"
    print(f"  {key} <- {info['role']}: {status}")

print("\nRole assignments done.")

  admin <- TENANT_ADMIN: assigned
  teacher <- SUBJECT_TEACHER: assigned
  creator <- CONTENT_CREATOR: assigned
  student1 <- STUDENT: assigned
  student2 <- STUDENT: assigned

Role assignments done.


In [41]:
# 3c. Login users to get fresh tokens
for key, info in data["users"].items():
    resp = requests.post(f"{AUTH_URL}/auth/login", json={
        "tenantId": str(TID),
        "identifier": info["email"],
        "password": info["password"]
    })
    if resp.status_code == 200:
        body = resp.json()
        data["users"][key]["accessToken"] = body["accessToken"]
        print(f"  {key}: logged in")
    else:
        print(f"  {key}: login failed {resp.status_code}")

print("\nAll users logged in.")

  admin: logged in
  teacher: logged in
  creator: logged in
  student1: logged in
  student2: logged in

All users logged in.


## Phase 4: Setup User Profiles & Class

In [42]:
# 4a. Create user profiles in user-profile-service (check first)
for key, info in data["users"].items():
    # Check if profile already exists
    check = requests.get(f"{PROFILE_URL}/v1/tenants/{TID}/profiles/by-user/{info['userId']}")
    if check.status_code == 200:
        print(f"  {key}: profile already exists")
        continue

    user_type = "STUDENT" if key.startswith("student") else "TEACHER" if key in ["teacher"] else "ADMIN"
    resp = requests.post(
        f"{PROFILE_URL}/v1/tenants/{TID}/profiles",
        json={
            "userId": info["userId"],
            "displayName": info["name"],
            "firstName": info["name"].split()[0],
            "lastName": info["name"].split()[-1] if len(info["name"].split()) > 1 else "",
            "email": info["email"],
            "userType": user_type,
            "status": "ACTIVE"
        },
        headers=h(user_id=info["userId"])
    )
    if resp.status_code in [200, 201]:
        print(f"  {key}: profile created")
    else:
        print(f"  {key}: {resp.status_code} - {resp.text[:100]}")

print("\nProfiles created.")

  admin: profile created
  teacher: profile created
  creator: profile created
  student1: profile created
  student2: profile created

Profiles created.


In [43]:
# 4b. Create a class in content-workflow-service (check first)
class_name = f"Class 10-A ({SUFFIX})"

# Check if class already exists
resp = requests.get(f"{WORKFLOW_URL}/classes", params={"grade": "10"}, headers=h(tenant_id=TID))
existing_class = None
if resp.status_code == 200:
    body = resp.json()
    items = body.get("items", []) if isinstance(body, dict) else body
    for c in items:
        if c.get("name") == class_name:
            existing_class = c
            break

if existing_class:
    data["classes"]["10A"] = str(existing_class["id"])
    CLASS_ID = data["classes"]["10A"]
    print(f"Class already exists: {CLASS_ID}")
else:
    resp = requests.post(
        f"{WORKFLOW_URL}/classes",
        json={
            "name": class_name,
            "subject": "Science",
            "grade": "10",
            "description": "Grade 10 Section A - Science"
        },
        headers=h(tenant_id=TID, user_id=data["users"]["teacher"]["userId"])
    )
    body = ok(resp, "Create Class")
    if body:
        data["classes"]["10A"] = str(body["id"])
    CLASS_ID = data["classes"]["10A"]

print(f"Class ID: {CLASS_ID}")

[PASS] Create Class -> 201
{
  "createdAt": "2026-02-08T12:24:00.958010077Z",
  "description": "Grade 10 Section A - Science",
  "grade": "10",
  "id": "29ef2b4f-7515-46a3-bd94-d4adbaa1236a",
  "name": "Class 10-A (837cda)",
  "releasedContent": [],
  "studentCount": 0,
  "students": [],
  "subject": "Science",
  "updatedAt": "2026-02-08T12:24:00.958011635Z"
}

Class ID: 29ef2b4f-7515-46a3-bd94-d4adbaa1236a


In [44]:
# 4c. Create student profiles with classId in user-profile-service
for key in ["student1", "student2"]:
    info = data["users"][key]
    resp = requests.put(
        f"{PROFILE_URL}/v1/tenants/{TID}/users/{info['userId']}/student-profile",
        json={
            "grade": "10",
            "section": "A",
            "rollNumber": key[-1],
            "classId": CLASS_ID,
            "board": "CBSE"
        },
        headers=h(user_id=info["userId"])
    )
    body = ok(resp, f"Student Profile {key}")

print("Student profiles with classId set.")

[PASS] Student Profile student1 -> 200
{
  "board": "CBSE",
  "classId": "29ef2b4f-7515-46a3-bd94-d4adbaa1236a",
  "createdAt": "2026-02-08T12:24:03.700438020Z",
  "grade": "10",
  "id": "7e54cfb0-b4e6-49f2-a081-d9956d6f025f",
  "rollNumber": "1",
  "section": "A",
  "tenantId": "1e20511f-422b-40fc-b443-ebbf90762edf",
  "updatedAt": "2026-02-08T12:24:03.700438020Z",
  "userId": "f63898d8-0eb1-4e42-bd3a-5f2e86cf3676"
}

[PASS] Student Profile student2 -> 200
{
  "board": "CBSE",
  "classId": "29ef2b4f-7515-46a3-bd94-d4adbaa1236a",
  "createdAt": "2026-02-08T12:24:03.717427584Z",
  "grade": "10",
  "id": "7d852165-26bb-492b-b4b8-cb392d307dde",
  "rollNumber": "2",
  "section": "A",
  "tenantId": "1e20511f-422b-40fc-b443-ebbf90762edf",
  "updatedAt": "2026-02-08T12:24:03.717427584Z",
  "userId": "2321b3e4-dc79-4366-8806-30c4de1e687a"
}

Student profiles with classId set.


In [45]:
# 4d. Verify students appear in class lookup
resp = requests.get(
    f"{PROFILE_URL}/v1/classes/{CLASS_ID}/students",
    headers=h(tenant_id=TID)
)
body = ok(resp, "Get Students in Class")
if body:
    print(f"Students found: {len(body.get('items', []))}")
    for s in body.get("items", []):
        print(f"  - {s.get('name', 'unknown')} ({s.get('id', '?')})")

[PASS] Get Students in Class -> 200
{
  "items": [
    {
      "avatar": null,
      "email": "student2_837cda@test.com",
      "grade": "10",
      "id": "2321b3e4-dc79-4366-8806-30c4de1e687a",
      "name": "Priya Singh",
      "rollNumber": "2",
      "section": "A"
    },
    {
      "avatar": null,
      "email": "student1_837cda@test.com",
      "grade": "10",
      "id": "f63898d8-0eb1-4e42-bd3a-5f2e86cf3676",
      "name": "Aarav Patel",
      "rollNumber": "1",
      "section": "A"
    }
  ]
}

Students found: 2
  - Priya Singh (2321b3e4-dc79-4366-8806-30c4de1e687a)
  - Aarav Patel (f63898d8-0eb1-4e42-bd3a-5f2e86cf3676)


## Phase 5: Content Creation & Workflow

In [46]:
# 5a. Content creator creates a note
creator = data["users"]["creator"]

resp = requests.post(
    f"{NOTES_URL}/notes",
    json={
        "title": f"Photosynthesis - Chapter 1 ({SUFFIX})",
        "summary": "Understanding the process of photosynthesis in plants.",
        "contentMd": "# Photosynthesis\n\nPhotosynthesis is the process by which green plants convert sunlight into food...\n\n## Key Concepts\n- Chlorophyll\n- Light reactions\n- Calvin cycle",
        "tags": ["#science", "#biology", "#photosynthesis", f"#test-{SUFFIX}"],
        "scopeType": "TENANT",
        "changeSummary": "Initial draft"
    },
    headers=h(token=creator["accessToken"], tenant_id=TID, user_id=creator["userId"])
)
body = ok(resp, "Create Note")
if body:
    data["notes"]["photosynthesis"] = {
        "id": body["id"],
        "versionId": body.get("latestVersionId"),
    }
    print(f"Note ID: {body['id']}")
    print(f"Version ID: {body.get('latestVersionId')}")

[PASS] Create Note -> 201
{
  "archiveReason": null,
  "archivedAt": null,
  "archivedBy": null,
  "createdAt": "2026-02-08T12:24:17.956398051Z",
  "createdBy": null,
  "deleted": false,
  "id": "86bc41eb-46b2-407b-90a4-9767ec0aff3f",
  "latestReleasedVersionId": null,
  "latestVersionId": "683ce516-9f86-4d88-9a0c-88adbb0b02fb",
  "rejectionReason": null,
  "releasedAt": null,
  "releasedBy": null,
  "reviewedAt": null,
  "reviewedBy": null,
  "scopeId": null,
  "scopeType": "TENANT",
  "status": "DRAFT",
  "submittedAt": null,
  "submittedBy": null,
  "summary": "Understanding the process of photosynthesis in plants.",
  "tags": [
    "#biology",
    "#photosynthesis",
    "#science",
    "#test-837cda"
  ],
  "tenantId": "1e20511f-422b-40fc-b443-ebbf90762edf",
  "title": "Photosynthesis - Chapter 1 (837cda)",
  "updatedAt": "2026-02-08T12:24:17.956398051Z",
  "updatedBy": null
}

Note ID: 86bc41eb-46b2-407b-90a4-9767ec0aff3f
Version ID: 683ce516-9f86-4d88-9a0c-88adbb0b02fb


In [47]:
# 5b. Create workflow for the note
note = data["notes"]["photosynthesis"]

resp = requests.post(
    f"{WORKFLOW_URL}/workflow",
    json={
        "contentId": note["id"],
        "contentVersionId": note["versionId"],
        "titleSnapshot": f"Photosynthesis - Chapter 1 ({SUFFIX})",
        "publishTargets": {
            "scope": "CLASS",
            "classIds": [CLASS_ID]
        }
    },
    headers=h(tenant_id=TID, user_id=creator["userId"])
)
body = ok(resp, "Create Workflow")
if body:
    data["workflows"]["photosynthesis"] = {
        "workflowId": body["workflowId"],
        "reviewTaskId": None
    }
    print(f"Workflow ID: {body['workflowId']}")
    print(f"State: {body['state']}")

[PASS] Create Workflow -> 201
{
  "contentId": "86bc41eb-46b2-407b-90a4-9767ec0aff3f",
  "contentVersionId": "683ce516-9f86-4d88-9a0c-88adbb0b02fb",
  "createdAt": "2026-02-08T12:24:21.431188554Z",
  "createdBy": "b78898d1-6bee-4f85-9b0f-940fda8b17d7",
  "currentStep": null,
  "lastUpdatedAt": "2026-02-08T12:24:21.431188554Z",
  "lastUpdatedBy": "b78898d1-6bee-4f85-9b0f-940fda8b17d7",
  "publishTargets": {
    "classIds": [
      "29ef2b4f-7515-46a3-bd94-d4adbaa1236a"
    ],
    "gradeIds": null,
    "scope": "CLASS",
    "sectionIds": null,
    "userIds": null
  },
  "requiredApprovals": null,
  "reviewTasks": [],
  "state": "DRAFT",
  "tenantId": "1e20511f-422b-40fc-b443-ebbf90762edf",
  "titleSnapshot": "Photosynthesis - Chapter 1 (837cda)",
  "version": 0,
  "workflowId": "2fa18368-daef-4a1c-bc41-b59190fa25f8"
}

Workflow ID: 2fa18368-daef-4a1c-bc41-b59190fa25f8
State: DRAFT


In [48]:
# 5c. Submit workflow for review (admin is the reviewer)
admin = data["users"]["admin"]
wf = data["workflows"]["photosynthesis"]

resp = requests.post(
    f"{WORKFLOW_URL}/workflow/{wf['workflowId']}/submit",
    json={
        "reviewerUserIds": [admin["userId"]],
        "requiredApprovals": 1,
        "note": "Please review for accuracy."
    },
    headers=h(tenant_id=TID, user_id=creator["userId"])
)
body = ok(resp, "Submit for Review")
if body:
    review_tasks = body.get("reviewTasks", [])
    if review_tasks:
        wf["reviewTaskId"] = review_tasks[0]["id"]
    print(f"State: {body['state']}")
    print(f"Review Task ID: {wf.get('reviewTaskId')}")

[PASS] Submit for Review -> 200
{
  "contentId": "86bc41eb-46b2-407b-90a4-9767ec0aff3f",
  "contentVersionId": "683ce516-9f86-4d88-9a0c-88adbb0b02fb",
  "createdAt": "2026-02-08T12:24:21.431189Z",
  "createdBy": "b78898d1-6bee-4f85-9b0f-940fda8b17d7",
  "currentStep": "REVIEW",
  "lastUpdatedAt": "2026-02-08T12:24:24.740311484Z",
  "lastUpdatedBy": "b78898d1-6bee-4f85-9b0f-940fda8b17d7",
  "publishTargets": {
    "classIds": [
      "29ef2b4f-7515-46a3-bd94-d4adbaa1236a"
    ],
    "gradeIds": null,
    "scope": "CLASS",
    "sectionIds": null,
    "userIds": null
  },
  "requiredApprovals": 1,
  "reviewTasks": [
    {
      "assigneeUserId": "05ecdf57-f283-45fc-b5d2-caedfaba135d",
      "comment": null,
      "createdAt": "2026-02-08T12:24:24.740311484Z",
      "id": "57c4c391-c540-4573-afed-b27d4eb61cb2",
      "status": "PENDING",
      "updatedAt": "2026-02-08T12:24:24.740311484Z",
      "workflowId": "2fa18368-daef-4a1c-bc41-b59190fa25f8"
    }
  ],
  "state": "IN_REVIEW",
  "tena

In [49]:
# 5d. Admin approves the review
wf = data["workflows"]["photosynthesis"]

resp = requests.post(
    f"{WORKFLOW_URL}/workflow/{wf['workflowId']}/reviews/{wf['reviewTaskId']}/approve",
    json={"comment": "Looks good. Approved."},
    headers=h(tenant_id=TID, user_id=admin["userId"])
)
body = ok(resp, "Approve Review")
if body:
    print(f"State: {body['state']}")

[PASS] Approve Review -> 200
{
  "contentId": "86bc41eb-46b2-407b-90a4-9767ec0aff3f",
  "contentVersionId": "683ce516-9f86-4d88-9a0c-88adbb0b02fb",
  "createdAt": "2026-02-08T12:24:21.431189Z",
  "createdBy": "b78898d1-6bee-4f85-9b0f-940fda8b17d7",
  "currentStep": null,
  "lastUpdatedAt": "2026-02-08T12:24:27.421517472Z",
  "lastUpdatedBy": "05ecdf57-f283-45fc-b5d2-caedfaba135d",
  "publishTargets": {
    "classIds": [
      "29ef2b4f-7515-46a3-bd94-d4adbaa1236a"
    ],
    "gradeIds": null,
    "scope": "CLASS",
    "sectionIds": null,
    "userIds": null
  },
  "requiredApprovals": 1,
  "reviewTasks": [
    {
      "assigneeUserId": "05ecdf57-f283-45fc-b5d2-caedfaba135d",
      "comment": "Looks good. Approved.",
      "createdAt": "2026-02-08T12:24:24.740311Z",
      "id": "57c4c391-c540-4573-afed-b27d4eb61cb2",
      "status": "APPROVED",
      "updatedAt": "2026-02-08T12:24:27.418084552Z",
      "workflowId": "2fa18368-daef-4a1c-bc41-b59190fa25f8"
    }
  ],
  "state": "APPROVED"

In [50]:
# 5e. Publish workflow
wf = data["workflows"]["photosynthesis"]

resp = requests.post(
    f"{WORKFLOW_URL}/workflow/{wf['workflowId']}/publish",
    json={
        "publishAt": None,
        "publishTargets": {
            "scope": "CLASS",
            "classIds": [CLASS_ID]
        }
    },
    headers=h(tenant_id=TID, user_id=admin["userId"])
)
body = ok(resp, "Publish Workflow")
if body:
    print(f"State: {body['state']}")

[PASS] Publish Workflow -> 200
{
  "contentId": "86bc41eb-46b2-407b-90a4-9767ec0aff3f",
  "contentVersionId": "683ce516-9f86-4d88-9a0c-88adbb0b02fb",
  "createdAt": "2026-02-08T12:24:21.431189Z",
  "createdBy": "b78898d1-6bee-4f85-9b0f-940fda8b17d7",
  "currentStep": null,
  "lastUpdatedAt": "2026-02-08T12:24:30.177825824Z",
  "lastUpdatedBy": "05ecdf57-f283-45fc-b5d2-caedfaba135d",
  "publishTargets": {
    "classIds": [
      "29ef2b4f-7515-46a3-bd94-d4adbaa1236a"
    ],
    "gradeIds": null,
    "scope": "CLASS",
    "sectionIds": null,
    "userIds": null
  },
  "requiredApprovals": 1,
  "reviewTasks": [
    {
      "assigneeUserId": "05ecdf57-f283-45fc-b5d2-caedfaba135d",
      "comment": "Looks good. Approved.",
      "createdAt": "2026-02-08T12:24:24.740311Z",
      "id": "57c4c391-c540-4573-afed-b27d4eb61cb2",
      "status": "APPROVED",
      "updatedAt": "2026-02-08T12:24:27.418085Z",
      "workflowId": "2fa18368-daef-4a1c-bc41-b59190fa25f8"
    }
  ],
  "state": "PUBLISHED"

In [51]:
# 5f. Release content to class (so students can see it)
note = data["notes"]["photosynthesis"]

resp = requests.post(
    f"{WORKFLOW_URL}/workflow/release",
    json={
        "contentIds": [note["id"]],
        "classIds": [CLASS_ID],
        "contentType": "note",
        "contentTitle": f"Photosynthesis - Chapter 1 ({SUFFIX})"
    },
    headers=h(tenant_id=TID, user_id=admin["userId"])
)
ok(resp, "Release Content to Class", expected=[200, 201, 204])
print("Content released to class.")

[PASS] Release Content to Class -> 200


Content released to class.


## Phase 6: Assignment Workflow

In [52]:
# 6a. Teacher creates an assignment (note -> class)
teacher = data["users"]["teacher"]
note = data["notes"]["photosynthesis"]
due_date = (datetime.utcnow() + timedelta(days=7)).strftime("%Y-%m-%dT23:59:59Z")

resp = requests.post(
    f"{WORKFLOW_URL}/assignments",
    json={
        "noteId": note["id"],
        "classId": CLASS_ID,
        "title": f"Photosynthesis Review Assignment ({SUFFIX})",
        "description": "Study the photosynthesis chapter and complete all flashcards.",
        "dueDate": due_date,
        "cycleConfig": {
            "intervalDays": [1, 3, 7, 14, 30],
            "autoRemind": True,
            "reminderHoursBefore": 24
        }
    },
    headers=h(tenant_id=TID, user_id=teacher["userId"])
)
body = ok(resp, "Create Assignment")
if body:
    data["assignments"]["photosynthesis"] = body["assignmentId"]
    print(f"Assignment ID: {body['assignmentId']}")
    print(f"Student Count: {body['studentCount']}")
    print(f"Status: {body['status']}")
    print(f"Due Date: {body['dueDate']}")

[PASS] Create Assignment -> 201
{
  "assignmentId": "9de5b6b9-589d-4d99-b54b-277d790199bd",
  "classId": "29ef2b4f-7515-46a3-bd94-d4adbaa1236a",
  "createdAt": "2026-02-08T12:24:34.855298433Z",
  "cycleConfig": {
    "autoRemind": true,
    "intervalDays": [
      1,
      3,
      7,
      14,
      30
    ],
    "reminderHoursBefore": 24
  },
  "description": "Study the photosynthesis chapter and complete all flashcards.",
  "dueDate": "2026-02-15T23:59:59Z",
  "noteId": "86bc41eb-46b2-407b-90a4-9767ec0aff3f",
  "status": "ACTIVE",
  "studentCount": 2,
  "teacherId": "61258019-de92-4578-a619-c2500355b642",
  "tenantId": "1e20511f-422b-40fc-b443-ebbf90762edf",
  "title": "Photosynthesis Review Assignment (837cda)",
  "updatedAt": "2026-02-08T12:24:34.855298433Z",
  "version": 0
}

Assignment ID: 9de5b6b9-589d-4d99-b54b-277d790199bd
Student Count: 2
Status: ACTIVE
Due Date: 2026-02-15T23:59:59Z


In [53]:
# 6b. List all assignments
resp = requests.get(
    f"{WORKFLOW_URL}/assignments",
    headers=h(tenant_id=TID)
)
body = ok(resp, "List Assignments")
if body:
    print(f"Total assignments: {body['totalElements']}")
    for a in body["items"]:
        print(f"  - {a['title']} (status={a['status']}, students={a['studentCount']})")

[PASS] List Assignments -> 200
{
  "items": [
    {
      "assignmentId": "9de5b6b9-589d-4d99-b54b-277d790199bd",
      "classId": "29ef2b4f-7515-46a3-bd94-d4adbaa1236a",
      "createdAt": "2026-02-08T12:24:34.855298Z",
      "cycleConfig": {
        "autoRemind": true,
        "intervalDays": [
          1,
          3,
          7,
          14,
          30
        ],
        "reminderHoursBefore": 24
      },
      "description": "Study the photosynthesis chapter and complete all flashcards.",
      "dueDate": "2026-02-15T23:59:59Z",
      "noteId": "86bc41eb-46b2-407b-90a4-9767ec0aff3f",
      "status": "ACTIVE",
      "studentCount": 2,
      "teacherId": "61258019-de92-4578-a619-c2500355b642",
      "tenantId": "1e20511f-422b-40fc-b443-ebbf90762edf",
      "title": "Photosynthesis Review Assignment (837cda)",
      "updatedAt": "2026-02-08T12:24:34.855298Z",
      "version": 1
    }
  ],
  "page": 0,
  "size": 20,
  "totalElements": 1,
  "totalPages": 1
}

Total assignments: 1


In [54]:
# 6c. Get assignment details
assignment_id = data["assignments"]["photosynthesis"]

resp = requests.get(
    f"{WORKFLOW_URL}/assignments/{assignment_id}",
    headers=h(tenant_id=TID)
)
ok(resp, "Get Assignment Details")

[PASS] Get Assignment Details -> 200
{
  "assignmentId": "9de5b6b9-589d-4d99-b54b-277d790199bd",
  "classId": "29ef2b4f-7515-46a3-bd94-d4adbaa1236a",
  "createdAt": "2026-02-08T12:24:34.855298Z",
  "cycleConfig": {
    "autoRemind": true,
    "intervalDays": [
      1,
      3,
      7,
      14,
      30
    ],
    "reminderHoursBefore": 24
  },
  "description": "Study the photosynthesis chapter and complete all flashcards.",
  "dueDate": "2026-02-15T23:59:59Z",
  "noteId": "86bc41eb-46b2-407b-90a4-9767ec0aff3f",
  "status": "ACTIVE",
  "studentCount": 2,
  "teacherId": "61258019-de92-4578-a619-c2500355b642",
  "tenantId": "1e20511f-422b-40fc-b443-ebbf90762edf",
  "title": "Photosynthesis Review Assignment (837cda)",
  "updatedAt": "2026-02-08T12:24:34.855298Z",
  "version": 1
}



{'assignmentId': '9de5b6b9-589d-4d99-b54b-277d790199bd',
 'classId': '29ef2b4f-7515-46a3-bd94-d4adbaa1236a',
 'createdAt': '2026-02-08T12:24:34.855298Z',
 'cycleConfig': {'autoRemind': True,
  'intervalDays': [1, 3, 7, 14, 30],
  'reminderHoursBefore': 24},
 'description': 'Study the photosynthesis chapter and complete all flashcards.',
 'dueDate': '2026-02-15T23:59:59Z',
 'noteId': '86bc41eb-46b2-407b-90a4-9767ec0aff3f',
 'status': 'ACTIVE',
 'studentCount': 2,
 'teacherId': '61258019-de92-4578-a619-c2500355b642',
 'tenantId': '1e20511f-422b-40fc-b443-ebbf90762edf',
 'title': 'Photosynthesis Review Assignment (837cda)',
 'updatedAt': '2026-02-08T12:24:34.855298Z',
 'version': 1}

In [55]:
# 6d. Get assignment progress (class-wide)
resp = requests.get(
    f"{WORKFLOW_URL}/assignments/{assignment_id}/progress",
    headers=h(tenant_id=TID)
)
body = ok(resp, "Get Assignment Progress")
if body:
    print(f"Total students: {body['totalStudents']}")
    print(f"Pending: {body['pendingCount']}, In Progress: {body['inProgressCount']}, Completed: {body['completedCount']}")
    for sa in body.get("studentAssignments", []):
        print(f"  - {sa['studentName']}: {sa['status']}")

[PASS] Get Assignment Progress -> 200
{
  "assignmentId": "9de5b6b9-589d-4d99-b54b-277d790199bd",
  "completedCount": 0,
  "inProgressCount": 0,
  "overdueCount": 0,
  "pendingCount": 2,
  "studentAssignments": [
    {
      "assignmentId": "9de5b6b9-589d-4d99-b54b-277d790199bd",
      "completedAt": null,
      "createdAt": "2026-02-08T12:24:34.855298Z",
      "recallScheduleId": null,
      "startedAt": null,
      "status": "PENDING",
      "studentAssignmentId": "2095b35f-0a5e-4971-a21a-aa2efae2ad08",
      "studentId": "2321b3e4-dc79-4366-8806-30c4de1e687a",
      "studentName": "Priya Singh",
      "updatedAt": "2026-02-08T12:24:34.855298Z"
    },
    {
      "assignmentId": "9de5b6b9-589d-4d99-b54b-277d790199bd",
      "completedAt": null,
      "createdAt": "2026-02-08T12:24:34.855298Z",
      "recallScheduleId": null,
      "startedAt": null,
      "status": "PENDING",
      "studentAssignmentId": "b2a3aa90-8fc1-48bd-bb3a-1ac548a12c69",
      "studentId": "f63898d8-0eb1-4e42-b

In [56]:
# 6e. Get assignments for a specific student
student1 = data["users"]["student1"]

resp = requests.get(
    f"{WORKFLOW_URL}/assignments/users/{student1['userId']}",
    headers=h(tenant_id=TID)
)
body = ok(resp, "Get Student1 Assignments")
if body:
    print(f"Student1 has {body['totalElements']} assignment(s)")

[PASS] Get Student1 Assignments -> 200
{
  "items": [
    {
      "assignmentId": "9de5b6b9-589d-4d99-b54b-277d790199bd",
      "classId": "29ef2b4f-7515-46a3-bd94-d4adbaa1236a",
      "createdAt": "2026-02-08T12:24:34.855298Z",
      "cycleConfig": {
        "autoRemind": true,
        "intervalDays": [
          1,
          3,
          7,
          14,
          30
        ],
        "reminderHoursBefore": 24
      },
      "description": "Study the photosynthesis chapter and complete all flashcards.",
      "dueDate": "2026-02-15T23:59:59Z",
      "noteId": "86bc41eb-46b2-407b-90a4-9767ec0aff3f",
      "status": "ACTIVE",
      "studentCount": 2,
      "teacherId": "61258019-de92-4578-a619-c2500355b642",
      "tenantId": "1e20511f-422b-40fc-b443-ebbf90762edf",
      "title": "Photosynthesis Review Assignment (837cda)",
      "updatedAt": "2026-02-08T12:24:34.855298Z",
      "version": 1
    }
  ],
  "page": 0,
  "size": 20,
  "totalElements": 1,
  "totalPages": 1
}

Student1 has 

In [57]:
# 6f. Update assignment (extend due date)
new_due = (datetime.utcnow() + timedelta(days=14)).strftime("%Y-%m-%dT23:59:59Z")

resp = requests.patch(
    f"{WORKFLOW_URL}/assignments/{assignment_id}",
    json={"dueDate": new_due, "title": f"Photosynthesis Review - Extended ({SUFFIX})"},
    headers=h(tenant_id=TID, user_id=teacher["userId"])
)
body = ok(resp, "Update Assignment")
if body:
    print(f"New title: {body['title']}")
    print(f"New due date: {body['dueDate']}")

[PASS] Update Assignment -> 200
{
  "assignmentId": "9de5b6b9-589d-4d99-b54b-277d790199bd",
  "classId": "29ef2b4f-7515-46a3-bd94-d4adbaa1236a",
  "createdAt": "2026-02-08T12:24:34.855298Z",
  "cycleConfig": {
    "autoRemind": true,
    "intervalDays": [
      1,
      3,
      7,
      14,
      30
    ],
    "reminderHoursBefore": 24
  },
  "description": "Study the photosynthesis chapter and complete all flashcards.",
  "dueDate": "2026-02-22T23:59:59Z",
  "noteId": "86bc41eb-46b2-407b-90a4-9767ec0aff3f",
  "status": "ACTIVE",
  "studentCount": 2,
  "teacherId": "61258019-de92-4578-a619-c2500355b642",
  "tenantId": "1e20511f-422b-40fc-b443-ebbf90762edf",
  "title": "Photosynthesis Review - Extended (837cda)",
  "updatedAt": "2026-02-08T12:24:50.007701462Z",
  "version": 1
}

New title: Photosynthesis Review - Extended (837cda)
New due date: 2026-02-22T23:59:59Z


In [58]:
# 6g. Test duplicate prevention
note = data["notes"]["photosynthesis"]

resp = requests.post(
    f"{WORKFLOW_URL}/assignments",
    json={
        "noteId": note["id"],
        "classId": CLASS_ID,
        "title": "Duplicate Test"
    },
    headers=h(tenant_id=TID, user_id=teacher["userId"])
)
ok(resp, "Duplicate Assignment (expect 409)", expected=[409])

[PASS] Duplicate Assignment (expect 409) -> 409
{
  "error": "CONFLICT",
  "message": "Active assignment already exists for this note in this class",
  "traceId": "c35251bb-c619-4a98-9142-d3e2d7731822"
}



{'error': 'CONFLICT',
 'message': 'Active assignment already exists for this note in this class',
 'traceId': 'c35251bb-c619-4a98-9142-d3e2d7731822'}

## Phase 7: Student Activity (Concept Progress & Activity Rings)

In [59]:
# 7a. Student1 records a reading session
student1 = data["users"]["student1"]
note = data["notes"]["photosynthesis"]

resp = requests.post(
    f"{PROFILE_URL}/v1/tenants/{TID}/users/{student1['userId']}/progress/concepts/reading-session",
    json={"noteId": note["id"]},
    headers=h()
)
ok(resp, "Student1: Reading Session")

[PASS] Student1: Reading Session -> 201
{
  "createdAt": "2026-02-08T12:24:57.497495539Z",
  "id": "f0a50935-af39-451b-a51d-403a057a3220",
  "lastExamPassedAt": null,
  "lastExamScore": null,
  "lastReviewedAt": "2026-02-08T12:24:57.497539849Z",
  "masteryAchievedAt": null,
  "masteryState": "LEARNING",
  "memoryStrength": 1.0,
  "noteId": "86bc41eb-46b2-407b-90a4-9767ec0aff3f",
  "passedExamCount": 0,
  "retention": 1.0,
  "totalExamAttempts": 0,
  "totalFlashcardsReviewed": 0,
  "totalReadingSessions": 1,
  "updatedAt": "2026-02-08T12:24:57.497539849Z",
  "userId": "f63898d8-0eb1-4e42-bd3a-5f2e86cf3676"
}



{'createdAt': '2026-02-08T12:24:57.497495539Z',
 'id': 'f0a50935-af39-451b-a51d-403a057a3220',
 'lastExamPassedAt': None,
 'lastExamScore': None,
 'lastReviewedAt': '2026-02-08T12:24:57.497539849Z',
 'masteryAchievedAt': None,
 'masteryState': 'LEARNING',
 'memoryStrength': 1.0,
 'noteId': '86bc41eb-46b2-407b-90a4-9767ec0aff3f',
 'passedExamCount': 0,
 'retention': 1.0,
 'totalExamAttempts': 0,
 'totalFlashcardsReviewed': 0,
 'totalReadingSessions': 1,
 'updatedAt': '2026-02-08T12:24:57.497539849Z',
 'userId': 'f63898d8-0eb1-4e42-bd3a-5f2e86cf3676'}

In [60]:
# 7b. Student1 records a flashcard review
resp = requests.post(
    f"{PROFILE_URL}/v1/tenants/{TID}/users/{student1['userId']}/progress/concepts/flashcard-review",
    json={"noteId": note["id"]},
    headers=h()
)
ok(resp, "Student1: Flashcard Review")

[PASS] Student1: Flashcard Review -> 201
{
  "createdAt": "2026-02-08T12:24:57.497496Z",
  "id": "f0a50935-af39-451b-a51d-403a057a3220",
  "lastExamPassedAt": null,
  "lastExamScore": null,
  "lastReviewedAt": "2026-02-08T12:24:59.407607379Z",
  "masteryAchievedAt": null,
  "masteryState": "LEARNING",
  "memoryStrength": 1.0,
  "noteId": "86bc41eb-46b2-407b-90a4-9767ec0aff3f",
  "passedExamCount": 0,
  "retention": 1.0,
  "totalExamAttempts": 0,
  "totalFlashcardsReviewed": 1,
  "totalReadingSessions": 1,
  "updatedAt": "2026-02-08T12:24:59.407607379Z",
  "userId": "f63898d8-0eb1-4e42-bd3a-5f2e86cf3676"
}



{'createdAt': '2026-02-08T12:24:57.497496Z',
 'id': 'f0a50935-af39-451b-a51d-403a057a3220',
 'lastExamPassedAt': None,
 'lastExamScore': None,
 'lastReviewedAt': '2026-02-08T12:24:59.407607379Z',
 'masteryAchievedAt': None,
 'masteryState': 'LEARNING',
 'memoryStrength': 1.0,
 'noteId': '86bc41eb-46b2-407b-90a4-9767ec0aff3f',
 'passedExamCount': 0,
 'retention': 1.0,
 'totalExamAttempts': 0,
 'totalFlashcardsReviewed': 1,
 'totalReadingSessions': 1,
 'updatedAt': '2026-02-08T12:24:59.407607379Z',
 'userId': 'f63898d8-0eb1-4e42-bd3a-5f2e86cf3676'}

In [61]:
# 7c. Student1 records an exam result
resp = requests.post(
    f"{PROFILE_URL}/v1/tenants/{TID}/users/{student1['userId']}/progress/concepts/exam-result",
    json={"noteId": note["id"], "score": 85, "passed": True},
    headers=h()
)
ok(resp, "Student1: Exam Result (85%)")

[PASS] Student1: Exam Result (85%) -> 201
{
  "createdAt": "2026-02-08T12:24:57.497496Z",
  "id": "f0a50935-af39-451b-a51d-403a057a3220",
  "lastExamPassedAt": "2026-02-08T12:25:01.551536177Z",
  "lastExamScore": 85,
  "lastReviewedAt": "2026-02-08T12:25:01.551536177Z",
  "masteryAchievedAt": "2026-02-08T12:25:01.551536177Z",
  "masteryState": "MASTERY",
  "memoryStrength": 1.05,
  "noteId": "86bc41eb-46b2-407b-90a4-9767ec0aff3f",
  "passedExamCount": 1,
  "retention": 1.0,
  "totalExamAttempts": 1,
  "totalFlashcardsReviewed": 1,
  "totalReadingSessions": 1,
  "updatedAt": "2026-02-08T12:25:01.551536177Z",
  "userId": "f63898d8-0eb1-4e42-bd3a-5f2e86cf3676"
}



{'createdAt': '2026-02-08T12:24:57.497496Z',
 'id': 'f0a50935-af39-451b-a51d-403a057a3220',
 'lastExamPassedAt': '2026-02-08T12:25:01.551536177Z',
 'lastExamScore': 85,
 'lastReviewedAt': '2026-02-08T12:25:01.551536177Z',
 'masteryAchievedAt': '2026-02-08T12:25:01.551536177Z',
 'masteryState': 'MASTERY',
 'memoryStrength': 1.05,
 'noteId': '86bc41eb-46b2-407b-90a4-9767ec0aff3f',
 'passedExamCount': 1,
 'retention': 1.0,
 'totalExamAttempts': 1,
 'totalFlashcardsReviewed': 1,
 'totalReadingSessions': 1,
 'updatedAt': '2026-02-08T12:25:01.551536177Z',
 'userId': 'f63898d8-0eb1-4e42-bd3a-5f2e86cf3676'}

In [62]:
# 7d. Check activity rings for today
resp = requests.get(
    f"{PROFILE_URL}/v1/tenants/{TID}/users/{student1['userId']}/progress/activity-rings/today"
)
body = ok(resp, "Student1: Today's Activity Rings")
if body:
    print(f"Reading ring closed: {body.get('readingRingClosed')}")
    print(f"Flashcard ring closed: {body.get('flashcardRingClosed')}")
    print(f"Exam ring closed: {body.get('examRingClosed')}")
    print(f"All rings closed: {body.get('allRingsClosed')}")

[PASS] Student1: Today's Activity Rings -> 200
{
  "activityDate": "2026-02-08",
  "allRingsClosed": true,
  "createdAt": "2026-02-08T12:24:57.501816Z",
  "examCount": 1,
  "examRingClosed": true,
  "flashcardCount": 1,
  "flashcardRingClosed": true,
  "id": "f49aa54c-34c9-4b6a-933e-f62a3b858fd6",
  "readingCount": 1,
  "readingRingClosed": true,
  "updatedAt": "2026-02-08T12:25:01.554199Z",
  "userId": "f63898d8-0eb1-4e42-bd3a-5f2e86cf3676"
}

Reading ring closed: True
Flashcard ring closed: True
Exam ring closed: True
All rings closed: True


In [63]:
# 7e. Check streak info
resp = requests.get(
    f"{PROFILE_URL}/v1/tenants/{TID}/users/{student1['userId']}/progress/activity-rings/streak"
)
body = ok(resp, "Student1: Streak Info")
if body:
    print(f"Activity streak: {body.get('currentActivityStreak')} days")
    print(f"All-rings streak: {body.get('currentAllRingsStreak')} days")

[PASS] Student1: Streak Info -> 200
{
  "asOfDate": "2026-02-08",
  "currentActivityStreak": 1,
  "currentAllRingsStreak": 1,
  "userId": "f63898d8-0eb1-4e42-bd3a-5f2e86cf3676"
}

Activity streak: 1 days
All-rings streak: 1 days


In [64]:
# 7f. Check concept progress
resp = requests.get(
    f"{PROFILE_URL}/v1/tenants/{TID}/users/{student1['userId']}/progress/concepts/{note['id']}"
)
body = ok(resp, "Student1: Concept Progress for Note")
if body:
    print(f"Mastery state: {body.get('masteryState')}")
    print(f"Exam attempts: {body.get('totalExamAttempts')}")
    print(f"Last score: {body.get('lastExamScore')}")

[PASS] Student1: Concept Progress for Note -> 200
{
  "createdAt": "2026-02-08T12:24:57.497496Z",
  "id": "f0a50935-af39-451b-a51d-403a057a3220",
  "lastExamPassedAt": "2026-02-08T12:25:01.551536Z",
  "lastExamScore": 85,
  "lastReviewedAt": "2026-02-08T12:25:01.551536Z",
  "masteryAchievedAt": "2026-02-08T12:25:01.551536Z",
  "masteryState": "MASTERY",
  "memoryStrength": 1.05,
  "noteId": "86bc41eb-46b2-407b-90a4-9767ec0aff3f",
  "passedExamCount": 1,
  "retention": 0.9998456909192168,
  "totalExamAttempts": 1,
  "totalFlashcardsReviewed": 1,
  "totalReadingSessions": 1,
  "updatedAt": "2026-02-08T12:25:01.551536Z",
  "userId": "f63898d8-0eb1-4e42-bd3a-5f2e86cf3676"
}

Mastery state: MASTERY
Exam attempts: 1
Last score: 85


In [65]:
# 7g. Check weekly summary
resp = requests.get(
    f"{PROFILE_URL}/v1/tenants/{TID}/users/{student1['userId']}/progress/activity-rings/weekly-summary"
)
body = ok(resp, "Student1: Weekly Summary")
if body:
    print(f"Days with activity: {body.get('daysWithActivity')}")
    print(f"Total reading: {body.get('totalReadingCount')}")
    print(f"Total flashcard: {body.get('totalFlashcardCount')}")
    print(f"Total exam: {body.get('totalExamCount')}")

[PASS] Student1: Weekly Summary -> 200
{
  "days": [
    {
      "activityDate": "2026-02-08",
      "allRingsClosed": true,
      "createdAt": "2026-02-08T12:24:57.501816Z",
      "examCount": 1,
      "examRingClosed": true,
      "flashcardCount": 1,
      "flashcardRingClosed": true,
      "id": "f49aa54c-34c9-4b6a-933e-f62a3b858fd6",
      "readingCount": 1,
      "readingRingClosed": true,
      "updatedAt": "2026-02-08T12:25:01.554199Z",
      "userId": "f63898d8-0eb1-4e42-bd3a-5f2e86cf3676"
    }
  ],
  "daysWithActivity": 1,
  "daysWithAllRingsClosed": 1,
  "from": "2026-02-02",
  "to": "2026-02-08",
  "totalExamCount": 1,
  "totalFlashcardCount": 1,
  "totalReadingCount": 1,
  "userId": "f63898d8-0eb1-4e42-bd3a-5f2e86cf3676"
}

Days with activity: 1
Total reading: 1
Total flashcard: 1
Total exam: 1


## Phase 8: Recall & Notifications

In [66]:
# 8a. Record a recall attempt
resp = requests.post(
    f"{RECALL_URL}/recall/attempts",
    json={
        "topicId": note["id"],
        "option": "GOOD",
        "timeSpentSeconds": 45
    },
    headers=h(tenant_id=TID, user_id=student1["userId"])
)
ok(resp, "Student1: Record Recall Attempt")

[PASS] Student1: Record Recall Attempt -> 201
{
  "easeFactor": 2.5,
  "intervalSeconds": 259200,
  "lastReviewedAt": "2026-02-08T12:25:19.837701027Z",
  "nextReviewAt": "2026-02-11T12:25:19.837701027Z",
  "option": "GOOD",
  "streak": 1,
  "topicId": "86bc41eb-46b2-407b-90a4-9767ec0aff3f"
}



{'easeFactor': 2.5,
 'intervalSeconds': 259200,
 'lastReviewedAt': '2026-02-08T12:25:19.837701027Z',
 'nextReviewAt': '2026-02-11T12:25:19.837701027Z',
 'option': 'GOOD',
 'streak': 1,
 'topicId': '86bc41eb-46b2-407b-90a4-9767ec0aff3f'}

In [67]:
# 8b. Get recall schedule
resp = requests.get(
    f"{RECALL_URL}/recall/schedule/{note['id']}",
    headers=h(tenant_id=TID, user_id=student1["userId"])
)
ok(resp, "Student1: Recall Schedule")

[PASS] Student1: Recall Schedule -> 200
{
  "easeFactor": 2.5,
  "intervalSeconds": 259200,
  "lapseCount": 0,
  "lastOption": "GOOD",
  "lastReviewedAt": "2026-02-08T12:25:19.837701Z",
  "nextReviewAt": "2026-02-11T12:25:19.837701Z",
  "streak": 1,
  "topicId": "86bc41eb-46b2-407b-90a4-9767ec0aff3f"
}



{'easeFactor': 2.5,
 'intervalSeconds': 259200,
 'lapseCount': 0,
 'lastOption': 'GOOD',
 'lastReviewedAt': '2026-02-08T12:25:19.837701Z',
 'nextReviewAt': '2026-02-11T12:25:19.837701Z',
 'streak': 1,
 'topicId': '86bc41eb-46b2-407b-90a4-9767ec0aff3f'}

In [68]:
# 8c. Get due recall items
resp = requests.get(
    f"{RECALL_URL}/recall/due?limit=50",
    headers=h(tenant_id=TID, user_id=student1["userId"])
)
ok(resp, "Student1: Due Recall Items")

[PASS] Student1: Due Recall Items -> 200
[]



[]

In [69]:
# 8d. Create a notification
resp = requests.post(
    f"{RECALL_URL}/v1/notifications",
    json={
        "userIds": [student1["userId"]],
        "type": "RECALL_DUE",
        "title": "Time to review!",
        "message": "Photosynthesis chapter is due for review.",
        "referenceId": note["id"],
        "referenceType": "TOPIC"
    },
    headers=h(tenant_id=TID)
)
ok(resp, "Create Notification")

[PASS] Create Notification -> 200
{
  "created": 1
}



{'created': 1}

In [70]:
# 8e. Get notifications for student
resp = requests.get(
    f"{RECALL_URL}/v1/notifications?page=0&size=20",
    headers=h(tenant_id=TID, user_id=student1["userId"])
)
body = ok(resp, "Student1: Get Notifications")
if body and isinstance(body, dict):
    content = body.get("content", body.get("items", []))
    print(f"Notifications: {len(content)}")
    for n in content[:3]:
        print(f"  - [{n.get('type')}] {n.get('title')}: {n.get('message')}")

[PASS] Student1: Get Notifications -> 200
{
  "content": [
    {
      "createdAt": "2026-02-08T12:25:27.022456Z",
      "id": "8cac679f-df38-483c-b9ce-6dfdeeae6d66",
      "message": "Photosynthesis chapter is due for review.",
      "read": false,
      "readAt": null,
      "referenceId": "86bc41eb-46b2-407b-90a4-9767ec0aff3f",
      "referenceType": "TOPIC",
      "title": "Time to review!",
      "type": "SYSTEM_ANNOUNCEMENT"
    }
  ],
  "empty": false,
  "first": true,
  "last": true,
  "number": 0,
  "numberOfElements": 1,
  "pageable": {
    "offset": 0,
    "pageNumber": 0,
    "pageSize": 20,
    "paged": true,
    "sort": {
      "empty": true,
      "sorted": false,
      "unsorted": true
    },
    "unpaged": false
  },
  "size": 20,
  "sort": {
    "empty": true,
    "sorted": false,
    "unsorted": true
  },
  "totalElements": 1,
  "totalPages": 1
}

Notifications: 1
  - [SYSTEM_ANNOUNCEMENT] Time to review!: Photosynthesis chapter is due for review.


In [71]:
# 8f. Get unread count
resp = requests.get(
    f"{RECALL_URL}/v1/notifications/unread/count",
    headers=h(tenant_id=TID, user_id=student1["userId"])
)
ok(resp, "Student1: Unread Notification Count")

[PASS] Student1: Unread Notification Count -> 200
{
  "count": 1
}



{'count': 1}

## Phase 9: Content Progress Tracking

In [ ]:
# 9a. Record content progress
resp = requests.post(
    f"{PROFILE_URL}/v1/tenants/{TID}/users/{student1['userId']}/progress/content",
    json={
        "eventType": "CONTENT_COMPLETED",
        "contentId": note["id"],
        "contentType": "NOTE",
        "progressPercent": 100,
        "score": 85.5,
        "timeSpentSeconds": 1200,
        "lastPosition": "end",
        "eventData": {}
    },
    headers=h()
)
ok(resp, "Student1: Record Content Progress")

In [73]:
# 9b. Get aggregate progress
resp = requests.get(
    f"{PROFILE_URL}/v1/tenants/{TID}/users/{student1['userId']}/progress"
)
body = ok(resp, "Student1: Aggregate Progress")
if body:
    print(f"Total time spent: {body.get('totalTimeSpentSeconds')}s")
    print(f"Notes completed: {body.get('totalNotesCompleted')}")
    print(f"Streak days: {body.get('streakDays')}")

[PASS] Student1: Aggregate Progress -> 200
{
  "averageQuizScore": null,
  "createdAt": "2026-02-08T12:25:35.449150512Z",
  "id": "08764803-b0cc-4f9c-ac40-47f9d1a6348c",
  "knowledgeScore": null,
  "lastActivityAt": null,
  "streakDays": 0,
  "totalFlashcardsReviewed": 0,
  "totalMindmapsCompleted": 0,
  "totalMindmapsViewed": 0,
  "totalNotesCompleted": 0,
  "totalNotesViewed": 0,
  "totalQuizzesAttempted": 0,
  "totalTimeSpentSeconds": 0,
  "updatedAt": "2026-02-08T12:25:35.449153419Z",
  "userId": "f63898d8-0eb1-4e42-bd3a-5f2e86cf3676"
}

Total time spent: 0s
Notes completed: 0
Streak days: 0


In [74]:
# 9c. Get content progress history
resp = requests.get(
    f"{PROFILE_URL}/v1/tenants/{TID}/users/{student1['userId']}/progress/content"
)
ok(resp, "Student1: Content Progress History")

[PASS] Student1: Content Progress History -> 200
[]



[]

## Phase 10: Cleanup (Optional)

Uncomment and run below cells to delete test data.

In [ ]:
# # 10a. Cancel assignment
# assignment_id = data["assignments"].get("photosynthesis")
# if assignment_id:
#     resp = requests.delete(
#         f"{WORKFLOW_URL}/assignments/{assignment_id}",
#         headers=h(tenant_id=TID)
#     )
#     ok(resp, "Cleanup: Cancel Assignment", expected=[204])

# # 10b. Delete note
# note_id = data["notes"].get("photosynthesis", {}).get("id")
# if note_id:
#     resp = requests.delete(
#         f"{NOTES_URL}/notes/{note_id}",
#         headers=h(token=data['users']['creator']['accessToken'], tenant_id=TID)
#     )
#     ok(resp, "Cleanup: Delete Note", expected=[200, 204])

# # 10c. Delete users
# for key, info in data["users"].items():
#     resp = requests.delete(f"{AUTH_URL}/auth/users/{info['userId']}")
#     print(f"  Delete {key}: {resp.status_code}")

# print("Cleanup done.")

## Test Results Summary

In [75]:
print("=" * 60)
print("  BRAIN SYSTEM E2E TEST RESULTS")
print("=" * 60)

passed = sum(1 for _, r, _ in data["results"] if r == "PASS")
failed = sum(1 for _, r, _ in data["results"] if r == "FAIL")
total = len(data["results"])

for label, result, status in data["results"]:
    icon = "  " if result == "PASS" else "  "
    print(f"  {icon} {label} [{status}]")

print("=" * 60)
print(f"  Total: {total}  |  Passed: {passed}  |  Failed: {failed}")
print("=" * 60)

print(f"\nTenant: {data['tenant_id']}")
print(f"Users: {', '.join(data['users'].keys())}")
print(f"Class: {data['classes'].get('10A', 'N/A')}")
print(f"Notes: {list(data['notes'].keys())}")
print(f"Assignments: {list(data['assignments'].keys())}")

  BRAIN SYSTEM E2E TEST RESULTS
     Create Tenant [200]
     Create Class [201]
     Student Profile student1 [200]
     Student Profile student2 [200]
     Get Students in Class [200]
     Create Note [201]
     Create Workflow [201]
     Submit for Review [200]
     Approve Review [200]
     Publish Workflow [200]
     Release Content to Class [200]
     Create Assignment [201]
     List Assignments [200]
     Get Assignment Details [200]
     Get Assignment Progress [200]
     Get Student1 Assignments [200]
     Update Assignment [200]
     Duplicate Assignment (expect 409) [409]
     Student1: Reading Session [201]
     Student1: Flashcard Review [201]
     Student1: Exam Result (85%) [201]
     Student1: Today's Activity Rings [200]
     Student1: Streak Info [200]
     Student1: Concept Progress for Note [200]
     Student1: Weekly Summary [200]
     Student1: Record Recall Attempt [201]
     Student1: Recall Schedule [200]
     Student1: Due Recall Items [200]
     Create Notif